In [1]:
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
import json
import pandas as pd
import time
from datetime import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

In [2]:
def get_otp_route(from_lat, from_lon, to_lat, to_lon, date_time=None, transport_modes=None, port=8080):
    """
    OTP(OpenTripPlanner)를 사용하여 경로를 검색합니다.
    
    Args:
        from_lat: 출발지 위도
        from_lon: 출발지 경도
        to_lat: 목적지 위도
        to_lon: 목적지 경도
        date_time: 출발 시간 (None이면 현재 시간 사용)
        transport_modes: 교통수단 목록 (None이면 도보+대중교통 사용)
        port: OTP 서버 포트 번호 (기본값: 8080)
    
    Returns:
        GraphQL 응답 데이터
    """
    # OTP GraphQL 엔드포인트 (포트 번호 동적 설정)
    url = f"http://localhost:{port}/otp/routers/default/index/graphql"
    
    # 날짜와 시간 분리
    if date_time is None:
        now = datetime.now()
        date = now.strftime("%Y-%m-%d")
        time = now.strftime("%H:%M") 
    else:
        dt = datetime.fromisoformat(date_time.replace('Z', '+00:00'))
        date = dt.strftime("%Y-%m-%d")
        time = dt.strftime("%H:%M")
    
    # 기본 교통수단 설정
    if transport_modes is None:
        transport_modes = [
            # {"mode": "WALK"},
            {"mode": "TRANSIT"}
        ]
    
    # GraphQL 쿼리 (두 번째 쿼리 형식 사용)
    query = """
    query ($from: InputCoordinates!, $to: InputCoordinates!, $date: String!, $time: String!, $transportModes: [TransportMode!]) {
        plan(
            from: $from
            to: $to
            date: $date
            time: $time
            transportModes: $transportModes
            numItineraries: 20
            searchWindow: 7200
            walkReluctance: 5.0
            transferPenalty: 0
            walkSpeed: 1.3
        ) {
            itineraries {
                startTime
                endTime
                duration
                walkDistance
                generalizedCost
                legs {
                    mode
                    startTime
                    endTime
                    duration
                    distance
                    generalizedCost     
                    from {
                        name
                        lat
                        lon
                        departureTime
                        arrivalTime
                        stop {
                            gtfsId        
                            id            
                            code          
                        }
                    }
                    to {
                        name
                        lat
                        lon
                        departureTime
                        arrivalTime
                        stop {
                            gtfsId        
                            id            
                            code          
                        }
                    }
                    route {
                        gtfsId
                        longName
                        shortName
                    }
                    trip {
                        gtfsId
                    }
                    legGeometry {
                        points
                    }
                }
            }
        }
    }
    """
    
    # 변수 설정
    variables = {
        "from": {
            "lat": from_lat,
            "lon": from_lon
        },
        "to": {
            "lat": to_lat,
            "lon": to_lon
        },
        "date": date,
        "time": time,
        "transportModes": transport_modes
    }
    
    # Session 설정
    session = requests.Session()
    retry = Retry(connect=3, backoff_factor=0.5)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    
    # 요청 보내기
    headers = {'Content-Type': 'application/json'}
    response = session.post(
        url, 
        headers=headers,
        json={
            'query': query,
            'variables': variables
        }
    )
    
    # 응답 처리
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

In [5]:
od_data = pd.read_csv('../data/otp/input/otp_od_input_filtered.csv')

In [6]:
od_data

,od_pair,o_stop_id,d_stop_id,o_lat,o_lon,d_lat,d_lon,departure_time,trip_count
0,10003_10661,10003,10661,37.35292,126.94574,37.463600,126.897560,20250217074728,13
1,10003_10700,10003,10700,37.35292,126.94574,37.452220,126.901570,20250217142818,14
2,10003_10718,10003,10718,37.35292,126.94574,37.483990,126.902460,20250218205508,10
3,10003_1451,10003,1451,37.35292,126.94574,37.443636,127.008010,20250221092229,13
4,10003_1456,10003,1456,37.35292,126.94574,37.394434,126.964112,20250217144646,11
...,...,...,...,...,...,...,...,...,...
1671263,9999_9558,9999,9558,37.47947,126.94544,37.483410,126.922050,20250217161716,25
1671264,9999_9635,9999,9635,37.47947,126.94544,37.490130,126.925710,20250217220124,15
1671265,9999_9679,9999,9679,37.47947,126.94544,37.490440,126.927620,20250217084807,9
1671266,9999_9695,9999,9695,37.47947,126.94544,37.484190,126.927790,20250217133810,61


In [49]:
from datetime import datetime

from_lat = 37.555525
from_lon = 126.972171
to_lat = 37.54904
to_lon = 126.86337

date_time_raw = 20250207155633

dt = datetime.strptime(str(date_time_raw), "%Y%m%d%H%M%S")
date_time = dt.isoformat()  # "2025-02-07T15:56:33"

result = get_otp_route(from_lat, from_lon, to_lat, to_lon, date_time)
print(json.dumps(result, indent=2))

{
  "data": {
    "plan": {
      "itineraries": [
        {
          "startTime": 1738911679000,
          "endTime": 1738913681000,
          "duration": 2002,
          "walkDistance": 335.21000000000004,
          "generalizedCost": 4665,
          "legs": [
            {
              "mode": "WALK",
              "startTime": 1738911679000,
              "endTime": 1738911935000,
              "duration": 256.0,
              "distance": 329.6,
              "generalizedCost": 1271,
              "from": {
                "name": "Origin",
                "lat": 37.555525,
                "lon": 126.972171,
                "departureTime": 1738911679000,
                "arrivalTime": 1738911679000,
                "stop": null
              },
              "to": {
                "name": "\uc22d\ub840\ubb38",
                "lat": 37.5583,
                "lon": 126.97329,
                "departureTime": 1738911935000,
                "arrivalTime": 1738911935000,
          

In [63]:
from datetime import datetime

timestamp_ms = 1738913681000
dt = datetime.fromtimestamp(timestamp_ms / 1000)

print(dt)  # 2025-02-07 15:41:19
print(dt.strftime("%Y-%m-%d %H:%M:%S"))  # 2025-02-07 15:41:19
print(dt.strftime("%Y%m%d%H%M%S"))  # 20250207154119


2025-02-07 16:34:41
2025-02-07 16:34:41
20250207163441


In [68]:
result['data']['plan']['itineraries'][0]['legs'][4]

{'mode': 'WALK',
 'startTime': 1738913676000,
 'endTime': 1738913681000,
 'duration': 5.0,
 'distance': 5.61,
 'generalizedCost': 23,
 'from': {'name': '영일고등학교.강서도서관',
  'lat': 37.54899,
  'lon': 126.86336,
  'departureTime': 1738913676000,
  'arrivalTime': 1738913676000,
  'stop': {'gtfsId': '1:BS_1100_115000076',
   'id': 'U3RvcDoxOkJTXzExMDBfMTE1MDAwMDc2',
   'code': None}},
 'to': {'name': 'Destination',
  'lat': 37.54904,
  'lon': 126.86337,
  'departureTime': 1738913681000,
  'arrivalTime': 1738913681000,
  'stop': None},
 'route': None,
 'trip': None,
 'legGeometry': {'points': 'exddF__ieWDSIC'}}

In [4]:
import pandas as pd
import numpy as np
import json

# =========================
# 1) OTP 경로를 7개 카테고리로 분류
# =========================
def classify_otp_itinerary(itinerary, gtx_route_ids=None):
    """
    OTP itinerary의 legs를 분석해서 7개 카테고리로 분류
    gtx_route_ids: GTX 노선 ID 리스트 (예: ['GTX-A', '290'] 등)
    """
    if gtx_route_ids is None:
        gtx_route_ids = ['GTX', '290']  # GTX 식별용
    
    modes = set()
    
    for leg in itinerary.get('legs', []):
        mode = leg.get('mode')
        
        if mode == 'BUS':
            modes.add('bus')
        
        elif mode in ['SUBWAY', 'RAIL', 'TRAM']:
            # GTX 여부 판별
            route_info = leg.get('route') or {}
            route_id = route_info.get('gtfsId', '') if isinstance(route_info, dict) else ''
            route_name = route_info.get('shortName', '') if isinstance(route_info, dict) else ''
            
            is_gtx = any(gtx_id in str(route_id) or gtx_id in str(route_name) 
                        for gtx_id in gtx_route_ids)
            
            if is_gtx:
                modes.add('gtx')
            else:
                modes.add('train')
    
    # 7개 카테고리로 매핑
    if not modes:
        return "walk_only"  # 도보만 있는 경우
    
    if modes == {"bus"}:
        return "bus_only"
    elif modes == {"train"}:
        return "train_only"
    elif modes == {"gtx"}:
        return "gtx_only"
    elif modes == {"bus", "train"}:
        return "bus+train"
    elif modes == {"bus", "gtx"}:
        return "bus+gtx"
    elif modes == {"train", "gtx"}:
        return "train+gtx"
    elif modes == {"bus", "train", "gtx"}:
        return "bus+train+gtx"
    else:
        return "unknown"


# =========================
# 2) OTP 결과 파싱 및 카테고리 분류
# =========================
def parse_otp_result(otp_response, gtx_route_ids=None):
    """
    OTP 응답에서 itinerary별 정보 추출 + 카테고리 분류
    """
    itineraries = otp_response.get('data', {}).get('plan', {}).get('itineraries', [])
    
    results = []
    for i, itin in enumerate(itineraries):
        results.append({
            'itinerary_idx': i,
            'duration': itin.get('duration'),
            'walkDistance': itin.get('walkDistance'),
            'generalizedCost': itin.get('generalizedCost'),
            'num_legs': len(itin.get('legs', [])),
            'num_transfers': len([l for l in itin.get('legs', []) if l.get('mode') not in ['WALK']]) - 1,
            'category': classify_otp_itinerary(itin, gtx_route_ids),
            'legs': itin.get('legs', [])
        })
    
    return results


# =========================
# 3) 확률 배정 함수
# =========================
def assign_probabilities(otp_routes, smartcard_share, method='weighted'):
    """
    OTP 경로들에 스마트카드 기반 확률 배정
    
    method:
        'equal' - 같은 카테고리 내 균등 배분
        'weighted' - generalizedCost 기반 가중 배분
    """
    categories = ['bus_only', 'train_only', 'gtx_only', 
                  'bus+train', 'bus+gtx', 'train+gtx', 'bus+train+gtx']
    
    # 스마트카드 분담률 계산
    total = smartcard_share.get('trip_count_sum', 1)
    mode_probs = {}
    for cat in categories:
        mode_probs[cat] = smartcard_share.get(cat, 0) / total if total > 0 else 0
    
    # 카테고리별로 경로 그룹화
    cat_routes = {cat: [] for cat in categories}
    for route in otp_routes:
        cat = route['category']
        if cat in cat_routes:
            cat_routes[cat].append(route)
    
    # 확률 배정
    for cat in categories:
        routes = cat_routes[cat]
        cat_prob = mode_probs[cat]
        
        if not routes or cat_prob == 0:
            continue
        
        if method == 'equal':
            # 균등 배분
            prob_each = cat_prob / len(routes)
            for r in routes:
                r['probability'] = prob_each
        
        elif method == 'weighted':
            # generalizedCost 기반 (낮을수록 좋음 → 높은 확률)
            costs = [r.get('generalizedCost', 9999) for r in routes]
            
            # utility = -cost (비용 낮을수록 utility 높음)
            utilities = [-c for c in costs]
            
            # softmax
            exp_utils = [np.exp(u / 1000) for u in utilities]  # scaling
            exp_sum = sum(exp_utils)
            
            for r, exp_u in zip(routes, exp_utils):
                r['probability'] = cat_prob * (exp_u / exp_sum) if exp_sum > 0 else 0
    
    # 확률 없는 경로는 0
    for route in otp_routes:
        if 'probability' not in route:
            route['probability'] = 0.0
    
    return otp_routes


# =========================
# 4) 전체 파이프라인 예시
# =========================
def process_od_pair(od_stop_pair, otp_response, smartcard_df, gtx_route_ids=None, method='weighted'):
    """
    하나의 OD pair에 대해 OTP 경로 확률 배정
    """
    # 스마트카드에서 해당 OD 분담률 가져오기
    od_data = smartcard_df[smartcard_df['od_stop_pair'] == od_stop_pair]
    
    if len(od_data) == 0:
        print(f"Warning: {od_stop_pair} not found in smartcard data")
        return None
    
    smartcard_share = od_data.iloc[0].to_dict()
    
    # OTP 결과 파싱
    otp_routes = parse_otp_result(otp_response, gtx_route_ids)
    
    if not otp_routes:
        print(f"Warning: No OTP routes for {od_stop_pair}")
        return None
    
    # 확률 배정
    otp_routes = assign_probabilities(otp_routes, smartcard_share, method=method)
    
    # 결과 정리
    result = {
        'od_stop_pair': od_stop_pair,
        'smartcard_total': smartcard_share['trip_count_sum'],
        'otp_route_count': len(otp_routes),
        'routes': []
    }
    
    for r in otp_routes:
        result['routes'].append({
            'itinerary_idx': r['itinerary_idx'],
            'category': r['category'],
            'duration': r['duration'],
            'num_transfers': r['num_transfers'],
            'generalizedCost': r['generalizedCost'],
            'probability': round(r['probability'], 4)
        })
    
    # 검증: 확률 합계
    prob_sum = sum(r['probability'] for r in result['routes'])
    result['probability_sum'] = round(prob_sum, 4)
    
    return result

In [9]:
# =========================
# 사용 예시
# =========================

# 스마트카드 데이터 로드 (이미 만든 것)
smartcard_df = pd.read_csv('../../multimodal_mobility_simulation_evaluation/data/trip_assignment/tcn_assignment_stop_7cat.csv')

# 위경도가 뒤바뀐 행 자동 수정
def fix_swapped_coords(df):
    """위경도가 뒤바뀐 행 자동 수정"""
    df = df.copy()
    
    # Origin 좌표 수정
    o_swapped = (df['o_lat'] > 100) | (df['o_lon'] < 100)
    df.loc[o_swapped, ['o_lat', 'o_lon']] = df.loc[o_swapped, ['o_lon', 'o_lat']].values
    
    # Destination 좌표 수정
    d_swapped = (df['d_lat'] > 100) | (df['d_lon'] < 100)
    df.loc[d_swapped, ['d_lat', 'd_lon']] = df.loc[d_swapped, ['d_lon', 'd_lat']].values
    
    print(f"Origin 수정: {o_swapped.sum()}건")
    print(f"Destination 수정: {d_swapped.sum()}건")
    
    return df

# 수정 적용
smartcard_df_fixed = fix_swapped_coords(smartcard_df)

# 확인
print(smartcard_df_fixed[['o_lat', 'o_lon', 'd_lat', 'd_lon']].describe())



# 거리별 필터링 (출발지 도착지 같은 경우)
def filter_valid_od(smartcard_df, min_distance=500, min_trips=2):
    """
    Haversine 공식으로 벡터 연산 (빠름)
    """
    df = smartcard_df.copy()
    
    # 라디안 변환
    lat1 = np.radians(df['o_lat'].values)
    lat2 = np.radians(df['d_lat'].values)
    lon1 = np.radians(df['o_lon'].values)
    lon2 = np.radians(df['d_lon'].values)
    
    # Haversine 공식
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # 지구 반지름 (미터)
    r = 6_371_000
    df['od_distance'] = c * r
    
    # 필터링
    filtered = df[
        (df['od_distance'] >= min_distance) &
        (df['trip_count_sum'] >= min_trips)
    ].copy()
    
    print(f"필터링 전: {len(smartcard_df)}")
    print(f"필터링 후: {len(filtered)}")
    print(f"제거된 OD: {len(smartcard_df) - len(filtered)}")
    
    return filtered

# 실행
smartcard_df_dis = filter_valid_od(smartcard_df_fixed, min_distance=500, min_trips=2)
smartcard_df_dis = smartcard_df_dis.reset_index(drop=True)

Origin 수정: 0건
Destination 수정: 1건
              o_lat         o_lon         d_lat         d_lon
count  7.469618e+06  7.469618e+06  7.469618e+06  7.469618e+06
mean   3.750967e+01  1.269558e+02  3.750593e+01  1.269583e+02
std    1.175672e-01  1.400438e-01  1.281841e-01  1.480385e-01
min    3.702692e+01  1.247194e+02  3.676968e+01  1.246513e+02
25%    3.746637e+01  1.268845e+02  3.745865e+01  1.268739e+02
50%    3.751768e+01  1.269771e+02  3.751644e+01  1.269775e+02
75%    3.757490e+01  1.270563e+02  3.757708e+01  1.270607e+02
max    3.800245e+01  1.274147e+02  3.821277e+01  1.277468e+02
필터링 전: 7469618
필터링 후: 3911372
제거된 OD: 3558246


In [10]:
smartcard_df_dis

,od_stop_pair,승차정류장ID,하차정류장ID,bus+gtx,bus+train,bus+train+gtx,bus_only,gtx_only,train+gtx,train_only,o_lat,o_lon,d_lat,d_lon,trip_count_sum,od_distance
0,10003_1006,10003,1006.0,0,4,0,0,0,0,0,37.35292,126.94574,37.515467,126.907650,4,18384.631762
1,10003_10661,10003,10661.0,0,0,0,13,0,0,0,37.35292,126.94574,37.463600,126.897560,13,13022.015904
2,10003_10681,10003,10681.0,0,0,0,2,0,0,0,37.35292,126.94574,37.476440,126.898920,2,14343.747786
3,10003_10700,10003,10700.0,0,0,0,16,0,0,0,37.35292,126.94574,37.452220,126.901570,16,11710.712250
4,10003_10718,10003,10718.0,0,0,0,11,0,0,0,37.35292,126.94574,37.483990,126.902460,11,15067.179796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3911367,9999_9692,9999,9692.0,0,0,0,5,0,0,0,37.47947,126.94544,37.498330,126.928250,5,2588.103296
3911368,9999_9695,9999,9695.0,0,0,0,64,0,0,0,37.47947,126.94544,37.484190,126.927790,64,1643.463441
3911369,9999_9773,9999,9773.0,0,0,0,5,0,0,0,37.47947,126.94544,37.500290,126.931690,5,2613.677821
3911370,9999_9871,9999,9871.0,0,0,0,74,0,0,0,37.47947,126.94544,37.475570,126.936420,74,906.425149


In [12]:
category_cols = ['bus_only', 'train_only', 'gtx_only', 'bus+train', 'bus+gtx', 'train+gtx', 'bus+train+gtx']

# num_categories 컬럼 생성 (0보다 큰 카테고리 수 카운트)
smartcard_df_dis['num_categories'] = (smartcard_df_dis[category_cols] > 0).sum(axis=1)

# 이제 필터링 가능
multi_category_od_3 = smartcard_df_dis[smartcard_df_dis['num_categories'] >= 3].copy()


In [15]:
# 3개 이상 카테고리가 있는 OD 추출
category_cols = ['bus_only', 'train_only', 'gtx_only', 'bus+train', 'bus+gtx', 'train+gtx', 'bus+train+gtx']

# num_categories 컬럼 생성 (0보다 큰 카테고리 수 카운트)
smartcard_df_dis['num_categories'] = (smartcard_df_dis[category_cols] > 0).sum(axis=1)


multi_category_od_3 = smartcard_df_dis[smartcard_df_dis['num_categories'] >= 3].copy()
print(f"3개 이상 카테고리 OD 수: {len(multi_category_od_3)}")

# 어떤 카테고리들이 섞여있는지 확인
multi_category_od_3['active_categories'] = multi_category_od_3.apply(
    lambda row: [col for col in category_cols if row[col] > 0], axis=1
)

# 샘플 확인
print("\n3개 이상 카테고리 OD 샘플:")
print(multi_category_od_3[['od_stop_pair', 'active_categories', 'trip_count_sum'] + category_cols].head(10))

# 테스트용 샘플 (통행 많은 순)
test_samples_3 = multi_category_od_3.nlargest(5, 'trip_count_sum')
print("\n테스트용 OD (3개 이상 카테고리):")
print(test_samples_3[['od_stop_pair', 'active_categories', 'trip_count_sum']])

# 테스트할 OD 리스트
test_od_list_3 = test_samples_3['od_stop_pair'].tolist()
test_od_list_3


3개 이상 카테고리 OD 수: 1844

3개 이상 카테고리 OD 샘플:
     od_stop_pair                                  active_categories  \
1049    1001_1276             [train_only, train+gtx, bus+train+gtx]   
1219    1001_1948             [train_only, bus+train, bus+train+gtx]   
1222    1001_1952             [train_only, bus+train, bus+train+gtx]   
1223    1001_1954             [train_only, train+gtx, bus+train+gtx]   
1224    1001_1955             [train_only, train+gtx, bus+train+gtx]   
1465     1001_309             [train_only, bus+train, bus+train+gtx]   
1466     1001_310             [train_only, bus+train, bus+train+gtx]   
2904    1001_<NA>  [train_only, bus+train, train+gtx, bus+train+gtx]   
3502    1002_1958                 [train_only, bus+train, train+gtx]   
3825     1002_309             [train_only, bus+train, bus+train+gtx]   

      trip_count_sum  bus_only  train_only  gtx_only  bus+train  bus+gtx  \
1049              45         0           1         0          0        0   
1219          

['312_1948', '1003_1275', '239_<NA>', '1275_1003', '239_1275']

In [156]:
test_od_list_3

['1854_1855', '1006_1801', '1954_1952', '311_1950', '424_239']

In [16]:
smartcard_df_dis[smartcard_df_dis['od_stop_pair'] == '312_1948']

,od_stop_pair,승차정류장ID,하차정류장ID,bus+gtx,bus+train,bus+train+gtx,bus_only,gtx_only,train+gtx,train_only,o_lat,o_lon,d_lat,d_lon,trip_count_sum,od_distance,num_categories
1222964,312_1948,312,1948.0,1,1,0,1453,0,0,0,37.610046,126.930267,37.650463,126.873933,1455,6693.901386,3


In [17]:
# OTP 응답 예시 (실제로는 API 호출 결과)
i = 1222964
from_lat = smartcard_df_dis.loc [i, 'o_lat']
from_lon = smartcard_df_dis.loc[i, 'o_lon']
to_lat = smartcard_df_dis.loc[i, 'd_lat']
to_lon = smartcard_df_dis.loc[i, 'd_lon']

otp_response = get_otp_route(from_lat, from_lon, to_lat, to_lon)
# print(json.dumps(otp_response, indent=2))

# 하나의 OD pair 처리
result = process_od_pair(
    od_stop_pair=smartcard_df_dis.loc[i, 'od_stop_pair'],
    otp_response=otp_response,
    smartcard_df=smartcard_df_dis,
    gtx_route_ids=['GTX', '290'],
    method='weighted'
)

print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "od_stop_pair": "312_1948",
  "smartcard_total": 1455,
  "otp_route_count": 3,
  "routes": [
    {
      "itinerary_idx": 0,
      "category": "bus_only",
      "duration": 2108,
      "num_transfers": 0,
      "generalizedCost": 4305,
      "probability": 0.3301
    },
    {
      "itinerary_idx": 1,
      "category": "bus_only",
      "duration": 2101,
      "num_transfers": 0,
      "generalizedCost": 4298,
      "probability": 0.3324
    },
    {
      "itinerary_idx": 2,
      "category": "bus_only",
      "duration": 2090,
      "num_transfers": 0,
      "generalizedCost": 4287,
      "probability": 0.3361
    }
  ],
  "probability_sum": 0.9986
}


In [18]:
def result_to_dataframe(result):
    """process_od_pair 결과를 DataFrame으로 변환"""
    if result is None:
        return pd.DataFrame()
    
    df = pd.DataFrame(result['routes'])
    df['od_stop_pair'] = result['od_stop_pair']
    df['smartcard_total'] = result['smartcard_total']
    
    # 컬럼 순서 정리
    cols = ['od_stop_pair', 'smartcard_total', 'itinerary_idx', 'category', 
            'duration', 'num_transfers', 'generalizedCost', 'probability']
    df = df[[c for c in cols if c in df.columns]]
    
    return df


result_df = result_to_dataframe(result)
result_df


,od_stop_pair,smartcard_total,itinerary_idx,category,duration,num_transfers,generalizedCost,probability
0,312_1948,1455,0,bus_only,2108,0,4305,0.3301
1,312_1948,1455,1,bus_only,2101,0,4298,0.3324
2,312_1948,1455,2,bus_only,2090,0,4287,0.3361


In [151]:
smartcard_df_dis

,od_stop_pair,승차정류장ID,하차정류장ID,bus+gtx,bus+train,bus+train+gtx,bus_only,gtx_only,train+gtx,train_only,o_lat,o_lon,d_lat,d_lon,trip_count_sum,od_distance
0,10003_1006,10003,1006.0,0,4,0,0,0,0,0,37.35292,126.94574,37.515467,126.907650,4,18384.631762
1,10003_10661,10003,10661.0,0,0,0,13,0,0,0,37.35292,126.94574,37.463600,126.897560,13,13022.015904
2,10003_10681,10003,10681.0,0,0,0,2,0,0,0,37.35292,126.94574,37.476440,126.898920,2,14343.747786
3,10003_10700,10003,10700.0,0,0,0,14,0,0,0,37.35292,126.94574,37.452220,126.901570,14,11710.712250
4,10003_10718,10003,10718.0,0,0,0,10,0,0,0,37.35292,126.94574,37.483990,126.902460,10,15067.179796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4197827,9999_9692,9999,9692.0,0,0,0,5,0,0,0,37.47947,126.94544,37.498330,126.928250,5,2588.103296
4197828,9999_9695,9999,9695.0,0,0,0,61,0,0,0,37.47947,126.94544,37.484190,126.927790,61,1643.463441
4197829,9999_9773,9999,9773.0,0,0,0,4,0,0,0,37.47947,126.94544,37.500290,126.931690,4,2613.677821
4197830,9999_9871,9999,9871.0,0,0,0,73,0,0,0,37.47947,126.94544,37.475570,126.936420,73,906.425149


In [152]:
smartcard_df_dis

,od_stop_pair,승차정류장ID,하차정류장ID,bus+gtx,bus+train,bus+train+gtx,bus_only,gtx_only,train+gtx,train_only,o_lat,o_lon,d_lat,d_lon,trip_count_sum,od_distance
0,10003_1006,10003,1006.0,0,4,0,0,0,0,0,37.35292,126.94574,37.515467,126.907650,4,18384.631762
1,10003_10661,10003,10661.0,0,0,0,13,0,0,0,37.35292,126.94574,37.463600,126.897560,13,13022.015904
2,10003_10681,10003,10681.0,0,0,0,2,0,0,0,37.35292,126.94574,37.476440,126.898920,2,14343.747786
3,10003_10700,10003,10700.0,0,0,0,14,0,0,0,37.35292,126.94574,37.452220,126.901570,14,11710.712250
4,10003_10718,10003,10718.0,0,0,0,10,0,0,0,37.35292,126.94574,37.483990,126.902460,10,15067.179796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4197827,9999_9692,9999,9692.0,0,0,0,5,0,0,0,37.47947,126.94544,37.498330,126.928250,5,2588.103296
4197828,9999_9695,9999,9695.0,0,0,0,61,0,0,0,37.47947,126.94544,37.484190,126.927790,61,1643.463441
4197829,9999_9773,9999,9773.0,0,0,0,4,0,0,0,37.47947,126.94544,37.500290,126.931690,4,2613.677821
4197830,9999_9871,9999,9871.0,0,0,0,73,0,0,0,37.47947,126.94544,37.475570,126.936420,73,906.425149
